<a href="https://colab.research.google.com/github/zettazet/willownutstea/blob/main/AI_Practice1_Test_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практичні завдання №1 (NumPy) — контрольне закріплення

**Формат роботи:** у кожному завданні потрібно реалізувати одну функцію **без циклів `for/while`** (використовуйте векторизацію NumPy), якщо в умові не сказано інакше.

Після реалізації функції запускайте клітинку **Перевірка** — вона покаже, чи все правильно.

**Оцінювання:** 10 завдань × 0,45 бала = **4,5 бала максимум**. Наприкінці є клітинка, що підраховує сумарний бал автоматично.


In [1]:
import numpy as np
import numpy.ma as ma

## Завдання 1. Нормалізація ознак (min-max) з підтримкою маскованих значень


У вас є одномірний масив значень `x`, у якому **деякі елементи можуть бути недійсними** і передані як **маскований масив** (`numpy.ma.MaskedArray`)
або як звичайний `np.ndarray`.

Реалізуйте функцію `minmax_scale(x)`, яка повертає масив тієї ж форми, де:

- коректні значення масштабуються в діапазон **[0, 1]** за формулою `(x - min) / (max - min)`;
- масковані значення **зберігають маску** (тобто залишаються маскованими);
- якщо `max == min` (усі коректні значення однакові), поверніть масив з нулями (для коректних значень) з тією ж маскою.

**Підказки:**
1. використовуйте `numpy.ma` функції агрегації.
2. для виділення маски з маскованого масиву скористайтесь функцією ma.getmaskarray(x)

In [2]:
def minmax_scale(x):
    """Min-max нормалізація з підтримкою MaskedArray."""
    m_x = ma.masked_array(x)

    x_min = m_x.min()
    x_max = m_x.max()

    if x_min is ma.masked or x_max == x_min:
        result = ma.zeros(m_x.shape)
        result.mask = m_x.mask
        return result
    scaled = (m_x - x_min) / (x_max - x_min)

    return scaled
    raise NotImplementedError

### Перевірка


In [3]:
def _test_minmax_scale():
    x = ma.array([10., 20., 30., 40.], mask=[0, 1, 0, 0])
    y = minmax_scale(x)
    assert isinstance(y, ma.MaskedArray), "Очікується MaskedArray на виході"
    assert y.mask.tolist() == [False, True, False, False]
    assert np.allclose(y[~y.mask], [0.0, 2/3, 1.0])

    x2 = np.array([5., 5., 5.])
    y2 = minmax_scale(x2)
    assert isinstance(y2, np.ndarray)
    assert np.allclose(y2, [0.0, 0.0, 0.0])

    x3 = ma.array([7., 7., 7.], mask=[1, 0, 1])
    y3 = minmax_scale(x3)
    assert y3.mask.tolist() == [True, False, True]
    assert float(y3[1]) == 0.0

_test_minmax_scale()
print("✅ Task 1 passed")

✅ Task 1 passed


## Завдання 2. Розумні зрізи: вирізати «вікно» та віддзеркалити по горизонталі


Дано 2D-масив `img` (наприклад, «зображення»). Реалізуйте:

`crop_and_flip(img, top, left, height, width)` → повертає фрагмент розміру `(height, width)`,
вирізаний з позиції `(top, left)` (верхній лівий кут), **і віддзеркалений по горизонталі** (ліворуч↔праворуч).

Використовуйте лише зрізи/індексацію NumPy (без циклів).


In [4]:
def crop_and_flip(img, top, left, height, width):
    """Вирізати прямокутник і віддзеркалити по горизонталі."""
    crop = img[top : top + height, left : left + width]
    return crop[:, ::-1]
    raise NotImplementedError

### Перевірка


In [5]:
def _test_crop_and_flip():
    img = np.arange(5*6).reshape(5,6)
    out = crop_and_flip(img, top=1, left=2, height=3, width=4)
    expected = img[1:4, 2:6][:, ::-1]
    assert out.shape == (3,4)
    assert np.array_equal(out, expected)

_test_crop_and_flip()
print("✅ Task 2 passed")

✅ Task 2 passed


## Завдання 3. Перетворення 1D сигналу в матрицю «дні × години» + добові середні


Є 1D-масив `readings` довжини `days*hours_per_day`, що містить погодинні вимірювання.

Реалізуйте `daily_mean(readings, days, hours_per_day)`, яка:
1) перетворює дані у матрицю форми `(days, hours_per_day)` у порядку запису,
2) повертає 1D-масив добових середніх значень довжини `days`.

Заборонено цикли.


In [6]:
def daily_mean(readings, days, hours_per_day):
    """Повертає середнє по кожному дню."""
    matrix = readings.reshape(days, hours_per_day)
    return matrix.mean(axis=1)
    raise NotImplementedError

### Перевірка


In [7]:
def _test_daily_mean():
    readings = np.arange(2*4)  # 2 дні, 4 години
    out = daily_mean(readings, days=2, hours_per_day=4)
    assert out.shape == (2,)
    assert np.allclose(out, [np.mean([0,1,2,3]), np.mean([4,5,6,7])])

    rng = np.random.default_rng(0)
    readings2 = rng.normal(size=3*24)
    out2 = daily_mean(readings2, 3, 24)
    assert np.allclose(out2, readings2.reshape(3,24).mean(axis=1))

_test_daily_mean()
print("✅ Task 3 passed")

✅ Task 3 passed


## Завдання 4. Збір фіч у єдину матрицю та спліт на train/test


Маємо:
- `features`: 2D масив форми `(n_samples, n_features)`
- `target`: 1D масив форми `(n_samples,)`

Реалізуйте `train_test_split_matrix(features, target, test_ratio=0.2)`, яка:
1) збирає **єдину матрицю** `Xy` форми `(n_samples, n_features+1)`, де останній стовпець — `target`;
2) робить розбиття **за індексом** (без перемішування): перші `n_train` рядків — train, решта — test,
де `n_test = ceil(n_samples * test_ratio)`, `n_train = n_samples - n_test`;
3) повертає `(train, test)` як дві матриці.

Використовуйте `np.hstack` або `np.concatenate`.


In [8]:
import math
def train_test_split_matrix(features, target, test_ratio=0.2):
    """Повертає train/test матриці з приєднаною ціллю."""
    target_column = target.reshape(-1, 1)
    Xy = np.hstack((features, target_column))

    n_samples = Xy.shape[0]
    n_test = math.ceil(n_samples * test_ratio)
    n_train = n_samples - n_test

    train = Xy[:n_train, :]
    test = Xy[n_train:, :]

    return train, test
    raise NotImplementedError

### Перевірка


In [9]:
def _test_train_test_split_matrix():
    features = np.arange(20).reshape(10,2)
    target = np.arange(10)
    train, test = train_test_split_matrix(features, target, test_ratio=0.2)
    assert train.shape == (8,3)
    assert test.shape == (2,3)
    assert np.array_equal(train[:, :2], features[:8])
    assert np.array_equal(train[:, 2], target[:8])
    assert np.array_equal(test[:, :2], features[8:])
    assert np.array_equal(test[:, 2], target[8:])

_test_train_test_split_matrix()
print("✅ Task 4 passed")

✅ Task 4 passed


## Завдання 5. Стандартизація колонок з broadcasting


Реалізуйте `standardize_columns(X)`, яка приймає 2D масив `X` форми `(n_samples, n_features)`
і повертає стандартизований масив `Z` такої ж форми:

`Z = (X - mean_per_col) / std_per_col`, де `mean_per_col` та `std_per_col` рахуються по `axis=0`.

Якщо для деякої колонки `std == 0`, то для цієї колонки у `Z` мають бути нулі (щоб уникнути ділення на 0).

Заборонено цикли.

std можна підраховувати з використанням функції np.std(x, axis=0)

Скористайтесь функцією np.divide(). Краще спочатку окремо створити масив відповідної розмірності, заповнений нулями, а потім застосувати цю функцію, вказавши параметри out і where.


In [10]:
def standardize_columns(X):
    """Колонкова стандартизація з обробкою std=0."""
    mean_per_col = np.mean(X, axis=0)
    std_per_col = np.std(X, axis=0)

    centered_X = X - mean_per_col
    Z = np.zeros_like(X)

    np.divide(centered_X, std_per_col, out=Z, where=(std_per_col != 0))
    return Z

    raise NotImplementedError

### Перевірка


In [11]:
def _test_standardize_columns():
    X = np.array([[1., 2., 3.],
                  [1., 5., 3.],
                  [1., 8., 3.]])
    Z = standardize_columns(X)
    # 1-ша і 3-тя колонки константні => нулі
    assert np.allclose(Z[:,0], 0)
    assert np.allclose(Z[:,2], 0)
    # 2-га колонка: mean=5, std=~2.449
    mean2 = X[:,1].mean()
    std2 = X[:,1].std()
    assert np.allclose(Z[:,1], (X[:,1]-mean2)/std2)

_test_standardize_columns()
print("✅ Task 5 passed")

✅ Task 5 passed


## Завдання 6. Агрегації по групах: середні значення по класах


Є:
- `X`: 2D масив ознак форми `(n_samples, n_features)`
- `labels`: 1D масив цілих міток класів довжини `n_samples` (наприклад 0..K-1)

Реалізуйте `class_means(X, labels, n_classes)` → 2D масив форми `(n_classes, n_features)`,
де рядок `k` — це середнє значення ознак для об’єктів класу `k`.

Обмеження:
- цикли по `n_samples` заборонені;
- цикли по `n_classes` **дозволені** (їх мало).


In [12]:
def class_means(X, labels, n_classes):
    """Середні значення ознак по кожному класу."""
    n_features = X.shape[1]
    means = np.zeros((n_classes, n_features))

    for k in range(n_classes):
        class_mask = (labels == k)
        class_samples = X[class_mask]
        if class_samples.shape[0] > 0:
            means[k] = class_samples.mean(axis=0)

    return means
    raise NotImplementedError

### Перевірка


In [13]:
def _test_class_means():
    X = np.array([[1., 10.],
                  [2., 20.],
                  [3., 30.],
                  [4., 40.]])
    labels = np.array([0, 1, 0, 1])
    out = class_means(X, labels, n_classes=2)
    assert out.shape == (2,2)
    assert np.allclose(out[0], np.mean(X[labels==0], axis=0))
    assert np.allclose(out[1], np.mean(X[labels==1], axis=0))

_test_class_means()
print("✅ Task 6 passed")

✅ Task 6 passed


## Завдання 7. Матричний добуток: обчислення cosine similarity матриця-матриця


Нехай `A` — матриця ембеддингів форми `(m, d)`, `B` — матриця форми `(n, d)`.

Реалізуйте `cosine_similarity_matrix(A, B)`, яка повертає матрицю `S` форми `(m, n)`,
де `S[i, j]` — cosine similarity між `A[i]` та `B[j]`.

Формула: `cos(a,b) = (a·b) / (||a|| * ||b||)`

Умови:
- використайте `np.dot`/`@` та broadcasting (без циклів);
- якщо вектор має нульову норму, вважайте схожість з ним рівною 0 (у відповідних рядках/стовпцях).


In [14]:
def cosine_similarity_matrix(A, B):
    """Cosine similarity для всіх пар рядків A та B."""
    dot_product = A @ B.T

    norm_A = np.linalg.norm(A, axis=1)
    norm_B = np.linalg.norm(B, axis=1)

    norms_matrix = np.outer(norm_A, norm_B)

    similarity = np.zeros_like(dot_product)

    np.divide(dot_product, norms_matrix, out=similarity, where=(norms_matrix != 0))

    return similarity
    raise NotImplementedError

### Перевірка


In [15]:
def _test_cosine_similarity_matrix():
    A = np.array([[1., 0.],
                  [0., 0.],
                  [1., 1.]])
    B = np.array([[1., 0.],
                  [1., 1.]])
    S = cosine_similarity_matrix(A, B)
    assert S.shape == (3,2)
    # (1,0) with (1,0) => 1
    assert np.isclose(S[0,0], 1.0)
    # zero vector => 0
    assert np.allclose(S[1], 0.0)
    # (1,1) with (1,0) => 1/sqrt(2)
    assert np.isclose(S[2,0], 1/np.sqrt(2))
    # (1,1) with (1,1) => 1
    assert np.isclose(S[2,1], 1.0)

_test_cosine_similarity_matrix()
print("✅ Task 7 passed")

✅ Task 7 passed


## Завдання 8. Перевірка оборотності та стабільний визначник через slogdet


Реалізуйте `invert_if_well_conditioned(A, det_threshold=1e-8)`, яка:
1) обчислює визначник **стабільно** через `np.linalg.slogdet`;
2) якщо `|det(A)| < det_threshold`, повертає `None`;
3) інакше повертає `np.linalg.inv(A)`.

Функція має працювати для квадратних матриць 2D.


In [16]:
def invert_if_well_conditioned(A, det_threshold=1e-8):
    """Повертає інверсію або None залежно від |det|."""
    res = np.linalg.det(A)
    if res < det_threshold:
      return None
    return np.linalg.inv(A)

    raise NotImplementedError

### Перевірка


In [17]:
def _test_invert_if_well_conditioned():
    A = np.array([[2., 1.],
                  [1., 2.]])
    invA = invert_if_well_conditioned(A, det_threshold=1e-8)
    assert invA is not None
    assert np.allclose(A @ invA, np.eye(2), atol=1e-8)

    B = np.array([[1., 2.],
                  [2., 4.]])  # det=0
    invB = invert_if_well_conditioned(B, det_threshold=1e-8)
    assert invB is None

_test_invert_if_well_conditioned()
print("✅ Task 8 passed")

✅ Task 8 passed


## Завдання 9. Умовне присвоєння: winsorize (обрізання викидів за перцентилями)


Іноді для стабілізації даних «викиди» обрізають по перцентилях.

Реалізуйте `winsorize(x, lower_q=0.05, upper_q=0.95)`, яка:
- знаходить `lo = quantile(x, lower_q)` та `hi = quantile(x, upper_q)` (через `np.quantile`);
- замінює значення < lo на lo, значення > hi на hi;
- повертає **новий масив** (не змінює вхідний).

Використовуйте `np.where` (або `np.clip` — теж можна).


In [18]:
def winsorize(x, lower_q=0.05, upper_q=0.95):
    """Обрізання значень по перцентилях."""
    lo = np.quantile(x, lower_q)
    hi = np.quantile(x, upper_q)

    return np.clip(x, lo, hi)
    raise NotImplementedError

### Перевірка


In [19]:
def _test_winsorize():
    rng = np.random.default_rng(1)
    x = rng.normal(size=1000)
    x2 = x.copy()
    y = winsorize(x, 0.1, 0.9)
    assert np.array_equal(x, x2), "Функція не повинна змінювати вхідний масив"
    lo = np.quantile(x, 0.1)
    hi = np.quantile(x, 0.9)
    assert y.min() >= lo - 1e-12
    assert y.max() <= hi + 1e-12
    # значення всередині інтервалу не мають змінитись
    mid = (x>=lo) & (x<=hi)
    assert np.allclose(y[mid], x[mid])

_test_winsorize()
print("✅ Task 9 passed")

✅ Task 9 passed


## Завдання 10. Умовний вибір: конвертація балів у літерні оцінки


Дано масив `scores` зі значеннями в діапазоні [0, 1].

Реалізуйте `grade_letters(scores)`, яка повертає масив рядків такої ж форми з оцінками:

- `A` якщо score ≥ 0.90
- `B` якщо 0.80 ≤ score < 0.90
- `C` якщо 0.70 ≤ score < 0.80
- `D` якщо 0.60 ≤ score < 0.70
- `F` якщо score < 0.60

Потрібно зробити **векторизовано** (без циклів) за допомогою `np.where` (каскадом) або `np.select`.


In [20]:
def grade_letters(scores):
    """Повертає літерні оцінки для масиву score ∈ [0,1]."""
    return np.where(scores >= 0.90, 'A',
          np.where(scores >= 0.80, 'B',
          np.where(scores >= 0.70, 'C',
          np.where(scores >= 0.60, 'D', 'F'))))
    raise NotImplementedError

### Перевірка


In [21]:
def _test_grade_letters():
    scores = np.array([0.2, 0.55, 0.6, 0.69, 0.7, 0.799, 0.8, 0.899, 0.9, 1.0])
    out = grade_letters(scores)
    expected = np.array(["F","F","D","D","C","C","B","B","A","A"])
    assert out.shape == scores.shape
    assert out.dtype.kind in {"U","S","O"}
    assert np.array_equal(out, expected)

_test_grade_letters()
print("✅ Task 10 passed")

✅ Task 10 passed


## Автоматичний підрахунок балів

Запустіть клітинку нижче. Вона спробує виконати всі тести ще раз і порахує сумарний бал.


In [22]:
def run_all_tests(verbose=True):
    tests = [
        ("Task 1", _test_minmax_scale),
        ("Task 2", _test_crop_and_flip),
        ("Task 3", _test_daily_mean),
        ("Task 4", _test_train_test_split_matrix),
        ("Task 5", _test_standardize_columns),
        ("Task 6", _test_class_means),
        ("Task 7", _test_cosine_similarity_matrix),
        ("Task 8", _test_invert_if_well_conditioned),
        ("Task 9", _test_winsorize),
        ("Task 10", _test_grade_letters),
    ]
    points_per_task = 0.45
    passed = 0
    details = []
    for name, fn in tests:
        try:
            fn()
            passed += 1
            details.append((name, True, "OK"))
        except Exception as e:
            details.append((name, False, f"{type(e).__name__}: {e}"))
            if verbose:
                print(f"❌ {name}: {type(e).__name__}: {e}")
    total = round(passed * points_per_task, 2)
    if verbose:
        print("\n" + "="*40)
        print(f"Пройдено: {passed}/{len(tests)}")
        print(f"Підсумковий бал: {total} / 4.5")
        print("="*40)
    return total, details

run_all_tests()


Пройдено: 10/10
Підсумковий бал: 4.5 / 4.5


(4.5,
 [('Task 1', True, 'OK'),
  ('Task 2', True, 'OK'),
  ('Task 3', True, 'OK'),
  ('Task 4', True, 'OK'),
  ('Task 5', True, 'OK'),
  ('Task 6', True, 'OK'),
  ('Task 7', True, 'OK'),
  ('Task 8', True, 'OK'),
  ('Task 9', True, 'OK'),
  ('Task 10', True, 'OK')])